On this entire project this file is executed by google colab, for more technical resources.

In [25]:
!pip install unsloth transformers accelerate bitsandbytes datasets

In [37]:
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth import UnslothTrainer, UnslothTrainingArguments
from transformers import AutoTokenizer

In [30]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=1024,
    load_in_4bit=True,
)

==((====))==  Unsloth 2025.11.6: Fast Mistral patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.1+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.5.1
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [32]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]
)

In [33]:
dataset = load_dataset("json", data_files={
    "train": "../DataEngineering/sft_train.json",
    "val": "../DataEngineering/sft_val.json",
})

def preprocess(batch):
    text = (
        f"Instruction: {batch['instruction']}\n"
        f"Input: {batch['input']}\n"
        f"Answer: {batch['output']}"
    )

    encoded = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=1024,
    )

    batch["input_ids"] = encoded["input_ids"]
    batch["attention_mask"] = encoded["attention_mask"]
    batch["labels"] = encoded["input_ids"]   # Causal LM

    return batch

dataset = dataset.map(preprocess)

Map:   0%|          | 0/1528 [00:00<?, ? examples/s]

Map:   0%|          | 0/170 [00:00<?, ? examples/s]

In [40]:
args = UnslothTrainingArguments(
    output_dir="../model/lora_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,

    logging_steps=50,
    save_steps=500,
    eval_steps=200,
)

trainer = UnslothTrainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
)

In [41]:
trainer.train()

model.save_pretrained("../model/final_lora_weights")
tokenizer.save_pretrained("../model/final_lora_weights")

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,528 | Num Epochs = 3 | Total steps = 288
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: muhammedsheikmubaris (muhammedsheikmubaris-indian-institute-of-information-tec) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
50,6.468100
100,4.289100
150,4.231000
200,4.221300
250,4.169600


train/epoch,▁▂▄▅▇█
train/global_step,▁▂▄▅▇█
train/grad_norm,█▁▄▅▅
train/learning_rate,█▆▅▃▁
train/loss,█▁▁▁▁
total_flos,2.0144660137220506e+17
train/epoch,3
train/global_step,288
train/grad_norm,0.21175
train/learning_rate,4e-05
train/loss,4.1696


('../model/final_lora_weights/tokenizer_config.json',
 '../model/final_lora_weights/special_tokens_map.json',
 '../model/final_lora_weights/chat_template.jinja',
 '../model/final_lora_weights/tokenizer.model',
 '../model/final_lora_weights/added_tokens.json',
 '../model/final_lora_weights/tokenizer.json')